# 🔬 Phase 5B1: Deterministic Greedy Control Methods (80 Tokens)

**Mục tiêu:** Đánh giá Greedy Decoding (`do_sample=False`) ở `max_new_tokens=80` trên 500 mẫu Test.

**Các phương pháp đánh giá:**
1. **Runner-up Early-Stop** (alpha=15.0, K=16, linear decay, Greedy decoding)
2. **Control: Random Direction** (v_rand, Greedy decoding)
3. **Control: Sign-Flipped** (-v_steer, Greedy decoding)

**Phạm vi:** 500 mẫu Test độc lập  
**Thời gian ước tính:** ~2 - 2.5 giờ  
**Output:** `phase5b1_control_80_greedy_results.json`

---

In [1]:
# Cell 1: Install Dependencies
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm
print('✅ Dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
✅ Dependencies installed!


In [2]:
# Cell 2: Environment & Config
import os, json, glob, random, time, math, gc
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR = '/kaggle/working'

ES_ALPHA = 15.0
ES_K = 16
ES_DECAY = 'linear'
MAX_TOKENS = 80

print(f'Config: alpha={ES_ALPHA}, K={ES_K}, decay={ES_DECAY}, max_tokens={MAX_TOKENS}')

Config: alpha=15.0, K=16, decay=linear, max_tokens=80


In [3]:
# Cell 3: Data Loading (Identical Split)
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}', f'./{DATA_FILENAME}'
]
data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches: data_path = matches[0]; break
if not data_path: raise FileNotFoundError(f'❌ {DATA_FILENAME} not found')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)
shuffled_records = list(raw_dataset)
random.seed(SEED)
random.shuffle(shuffled_records)
n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
test_records = shuffled_records[n_train + n_val:]
print(f'📊 Loaded Test Split: {len(test_records):,} records')

📊 Loaded Test Split: 2,205 records


In [4]:
# Cell 4: Load Phase 1 Artifacts
config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True)
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
v_rand_paths = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True)
if not config_paths: raise FileNotFoundError('❌ steering_config.json not found')

with open(config_paths[0], 'r') as f:
    steering_config = json.load(f)
BEST_LAYER = steering_config['best_layer']
v_steer = torch.load(v_steer_paths[0], map_location='cpu')
v_rand = torch.load(v_rand_paths[0], map_location='cpu') if v_rand_paths else None
print(f'✅ Layer: {BEST_LAYER} | v_steer shape: {v_steer.shape}')

✅ Layer: 8 | v_steer shape: torch.Size([3584])


In [5]:
# Cell 5: Load Model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'⌛ Loading {MODEL_NAME}...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model.eval()
print('✅ Model loaded!')

⌛ Loading Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded!


In [6]:
# Cell 6: Steering Hook & Engine
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8, decay='hard'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    
    def _eff_alpha(self, t):
        if self.K >= 999: return self.alpha
        if t >= self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'cosine':
            return self.alpha * 0.5 * (1.0 + math.cos(math.pi * t / self.K))
        elif self.decay == 'linear':
            return self.alpha * (1.0 - t / self.K)
        return self.alpha
    
    def hook_fn(self, module, inputs, output):
        a = self._eff_alpha(self.step_counter)
        if a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + a * v
        self.step_counter += 1
        return output
    
    def register(self, mdl):
        self.step_counter = 0
        self.handle = mdl.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    def remove(self):
        if self.handle: self.handle.remove(); self.handle = None

def evaluate_greedy(model, tokenizer, records, v_vector, alpha, K, decay,
                    max_new_tokens=80, name='', layer_idx=None):
    results = []
    t_start = time.time()
    for idx, rec in enumerate(tqdm(records, desc=name)):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        ref = rec['right_answer']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        hook = None
        if v_vector is not None and layer_idx is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        elapsed_ms = (time.time() - t0) * 1000
        if hook: hook.remove()
        
        gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
        eos_id = tokenizer.eos_token_id
        hit_eos = bool(len(gen_tokens) > 0 and gen_tokens[-1].item() == eos_id)
        words = gen_text.split()
        if len(words) >= 4:
            ngrams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
            rep4 = 1.0 - len(set(ngrams))/len(ngrams) if ngrams else 0.0
        else: rep4 = 0.0
        
        results.append({
            'idx': idx, 'rouge_l': r_score, 'num_tokens': len(gen_tokens),
            'elapsed_ms': elapsed_ms, 'hit_eos': hit_eos, 'rep_4gram': rep4,
            'generated': gen_text, 'reference': ref, 'question': q,
            'category': rec.get('hallucination_type', 'unknown'),
        })
    avg_rl = np.mean([r['rouge_l'] for r in results])
    total_min = (time.time() - t_start) / 60
    print(f'  ✅ [{name}] {total_min:.1f}min | ROUGE-L: {avg_rl:.2f}% | Rep4gram: {np.mean([r["rep_4gram"] for r in results]):.4f}')
    return results

print('✅ Greedy Engine ready!')

✅ Greedy Engine ready!


In [7]:
# Cell 7: Run Evaluation
print('='*70)
print('PHASE 5B1: DETERMINISTIC GREEDY CONTROL METHODS (80 TOKENS)')
print('='*70)

test_subset = test_records[:500]
results_dict = {}

print('\n--- [1/3] Runner ES (Greedy, max=80) ---')
results_dict['es_runner_80_greedy'] = evaluate_greedy(model, tokenizer, test_subset, v_vector=v_steer, alpha=ES_ALPHA, K=ES_K, decay=ES_DECAY, max_new_tokens=80, name='Runner ES Greedy', layer_idx=BEST_LAYER)

print('\n--- [2/3] Ctrl Random (Greedy, max=80) ---')
results_dict['ctrl_random_80_greedy'] = evaluate_greedy(model, tokenizer, test_subset, v_vector=v_rand, alpha=20.0, K=999, decay='hard', max_new_tokens=80, name='Ctrl Random Greedy', layer_idx=BEST_LAYER)

print('\n--- [3/3] Ctrl SignFlip (Greedy, max=80) ---')
results_dict['ctrl_signflip_80_greedy'] = evaluate_greedy(model, tokenizer, test_subset, v_vector=-v_steer, alpha=20.0, K=999, decay='hard', max_new_tokens=80, name='Ctrl SignFlip Greedy', layer_idx=BEST_LAYER)

gc.collect(); torch.cuda.empty_cache()

PHASE 5B1: DETERMINISTIC GREEDY CONTROL METHODS (80 TOKENS)

--- [1/3] Runner ES (Greedy, max=80) ---


Runner ES Greedy: 100%|██████████| 500/500 [1:21:16<00:00,  9.75s/it]


  ✅ [Runner ES Greedy] 81.3min | ROUGE-L: 34.23% | Rep4gram: 0.0117

--- [2/3] Ctrl Random (Greedy, max=80) ---


Ctrl Random Greedy: 100%|██████████| 500/500 [1:21:49<00:00,  9.82s/it]


  ✅ [Ctrl Random Greedy] 81.8min | ROUGE-L: 33.94% | Rep4gram: 0.0111

--- [3/3] Ctrl SignFlip (Greedy, max=80) ---


Ctrl SignFlip Greedy: 100%|██████████| 500/500 [1:21:41<00:00,  9.80s/it]


  ✅ [Ctrl SignFlip Greedy] 81.7min | ROUGE-L: 33.50% | Rep4gram: 0.0168


In [8]:
# Cell 8: BERTScore
print('\n' + '='*70)
print('COMPUTING BERTSCORE FOR GREEDY EVALUATION')
print('='*70)

try:
    from bert_score import score as bert_score_fn
    for cond, results in results_dict.items():
        refs = [r['reference'] for r in results]
        hyps = [r['generated'] for r in results]
        print(f'  Computing BERTScore cho {cond}...')
        P, R, F1 = bert_score_fn(hyps, refs, model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device='cuda')
        for r, bs in zip(results, F1.tolist()):
            r['bertscore_f1'] = bs
        print(f'    → {cond}: BERTScore = {np.mean(F1.tolist()):.4f}')
    print('✅ BERTScore completed.')
except Exception as e:
    print(f'⚠️ BERTScore error: {e}')


COMPUTING BERTSCORE FOR GREEDY EVALUATION
  Computing BERTScore cho es_runner_80_greedy...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    → ctrl_random_80_greedy: BERTScore = 0.7470
  Computing BERTScore cho ctrl_signflip_80_greedy...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    → ctrl_signflip_80_greedy: BERTScore = 0.7435
✅ BERTScore completed.


In [9]:
# Cell 9: Summary & Export
print('\n' + '='*85)
print('📊 PHASE 5B1: DETERMINISTIC GREEDY CONTROL METHODS (80 TOKENS) RESULTS SUMMARY')
print('='*85)

def fmt(results):
    return {
        'rouge_l': np.mean([r['rouge_l'] for r in results]),
        'bertscore': np.mean([r.get('bertscore_f1',0) for r in results]),
        'rep4': np.mean([r['rep_4gram'] for r in results]),
        'eos': np.mean([r['hit_eos'] for r in results])*100,
        'len': np.mean([r['num_tokens'] for r in results]),
        'lat': np.mean([r['elapsed_ms'] for r in results]),
    }

print(f'{"Method":<32} {"ROUGE-L%":>9} {"BERTSc":>7} {"Rep4":>6} {"EOS%":>5} {"Len":>5} {"ms":>7}')
print('-'*78)
for key in results_dict.keys():
    s = fmt(results_dict[key])
    print(f'{key:<32} {s["rouge_l"]:>8.2f}% {s["bertscore"]:>6.4f} {s["rep4"]:>5.4f} {s["eos"]:>4.1f} {s["len"]:>5.1f} {s["lat"]:>6.0f}')

export = {
    'experiment': 'Phase 5B1: Deterministic Greedy Control Methods (80 Tokens)',
    'results_summary': {k: fmt(v) for k, v in results_dict.items()},
}
with open(os.path.join(OUTPUT_DIR, 'phase5b1_control_80_greedy_results.json'), 'w') as f:
    json.dump(export, f, indent=2)

texts = {c: [{'idx':r['idx'],'generated':r['generated'],'reference':r['reference'],
              'rouge_l':r['rouge_l'],'category':r['category']} for r in rs]
         for c, rs in results_dict.items()}
with open(os.path.join(OUTPUT_DIR, 'phase5b1_control_80_greedy_generated_texts.json'), 'w', encoding='utf-8') as f:
    json.dump(texts, f, indent=2, ensure_ascii=False)

print(f'\n💾 Exported: phase5b1_control_80_greedy_results.json + phase5b1_control_80_greedy_generated_texts.json')
print('🎉 HOÀN THÀNH!')


📊 PHASE 5B1: DETERMINISTIC GREEDY CONTROL METHODS (80 TOKENS) RESULTS SUMMARY
Method                            ROUGE-L%  BERTSc   Rep4  EOS%   Len      ms
------------------------------------------------------------------------------
es_runner_80_greedy                 34.23% 0.7468 0.0117  0.0  80.0   9748
ctrl_random_80_greedy               33.94% 0.7470 0.0111  0.0  80.0   9814
ctrl_signflip_80_greedy             33.50% 0.7435 0.0168  0.0  80.0   9798

💾 Exported: phase5b1_control_80_greedy_results.json + phase5b1_control_80_greedy_generated_texts.json
🎉 HOÀN THÀNH!
